# MBG Reply Pipeline — Colab
**Process 509k reply tweets → sentiment + reply tree + analysis**

### Before you start:
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Fill in credentials in **Cell 3**
3. Run all cells top-to-bottom

### Pipeline stages:
| Stage | Script | Output |
|---|---|---|
| R1 | `r1_jsonl_to_csv.py` | `replies_raw.csv` |
| R2 | `r2_enrich_metadata.py` | `replies_enriched.csv` |
| R3 | `r3_add_depth.py` | `replies_depth.csv` |
| R4 | `r4_filter_text.py` | `replies_filtered.csv` |
| R5 | `r5_tag_language.py` | `replies_tagged.csv` |
| R6 | `r6_preprocess_text.py` | `replies_preprocessed.csv` |
| R7 | `r7_sentiment.py` | `replies_sentiment.csv` |
| R8 | *(inline)* | `reply_tree.csv`, `corpus_combined.csv` |
| R9 | *(inline)* | `reply_*.csv` (5 analysis files) |
| R10 | *(inline)* | Upload to DO Spaces |

**Note**: No relevance filtering — replies inherit context from parents

## Cell 1 — Mount Drive & Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, time

notebook_start_time = time.time()
os.environ["RUNTIME_MODE"] = "colab"

DRIVE_BASE = "/content/drive/MyDrive/mbg"
DATA_DIR = f"{DRIVE_BASE}/data"
REPLY_DIR = f"{DATA_DIR}/replies"
OUTPUT_DIR = f"{DATA_DIR}/output"
CODE_DIR = "/content/mbg-pipeline"
ANALYSIS_DIR = f"{REPLY_DIR}/analysis"

for d in [REPLY_DIR, OUTPUT_DIR, ANALYSIS_DIR]:
    os.makedirs(d, exist_ok=True)

print("✅ Drive mounted")
print(f"   Reply data: {REPLY_DIR}")
print(f"   Outputs:    {OUTPUT_DIR}")
print(f"   Analysis:   {ANALYSIS_DIR}")
print(f"   Code:       {CODE_DIR}")

## Cell 2 — Verify GPU

In [ ]:
import torch

assert torch.cuda.is_available(), "❌ No GPU — go to Runtime → Change runtime type → T4 GPU"
print(f"✅ GPU ready: {torch.cuda.get_device_name(0)}")
print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Cell 3 — Clone Codebase

In [ ]:
GITHUB_REPO = "https://github.com/FatwaArya/mbg-analysis"

import subprocess, sys

if not os.path.exists(CODE_DIR):
    print("Cloning repo...")
    subprocess.run(["git", "clone", GITHUB_REPO, CODE_DIR], check=True)
else:
    print("Repo exists — pulling latest...")
    subprocess.run(["git", "-C", CODE_DIR, "pull"], check=True)

if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

print(f"✅ Codebase ready at {CODE_DIR}")

## Cell 4 — Install Dependencies

In [ ]:
import subprocess

req_path = f"{CODE_DIR}/requirements.txt"
print("Installing dependencies (~3-5 min)...")
subprocess.run(["pip", "install", "-q", "-r", req_path], check=True)
print("✅ Dependencies installed")

## Cell 5 — Download Reply Data from DO Spaces

In [ ]:
REPLY_JSONL = f"{REPLY_DIR}/replies_all_dedup.jsonl"

if os.path.exists(REPLY_JSONL):
    import os
    size_mb = os.path.getsize(REPLY_JSONL) / 1e6
    print(f"✅ Reply data found: {size_mb:.0f}MB")
else:
    print("Downloading reply data from DO Spaces (~276MB, ~2-3 min)...")
    !pip install -q s3cmd
    
    from google.colab import userdata
    s3cfg = f'[default]\naccess_key = {userdata.get("DO_ACCESS_KEY")}\nsecret_key = {userdata.get("DO_SECRET_KEY")}\nhost_base = sgp1.digitaloceanspaces.com\nhost_bucket = %(bucket)s.sgp1.digitaloceanspaces.com\n'
    open('/root/.s3cfg', 'w').write(s3cfg)
    
    !s3cmd get s3://mbg-scraper-network-20260419071440/replies_all_dedup.jsonl {REPLY_JSONL}
    print(f"✅ Downloaded to {REPLY_JSONL}")

## Cell 6 — Download Parent Posts (for metadata enrichment)

In [ ]:
PARENTS_CSV = f"{OUTPUT_DIR}/tweets_relevant.csv"
PARENTS_SENTIMENT = f"{OUTPUT_DIR}/tweets_with_sentiment.csv"

if os.path.exists(PARENTS_CSV):
    import pandas as pd
    df = pd.read_csv(PARENTS_CSV)
    print(f"✅ Parent posts found: {len(df):,} rows")
else:
    print("❌ Parent posts not found")
    print(f"   Upload tweets_relevant.csv to {OUTPUT_DIR}")
    print("   Or run parent pipeline first")
    raise FileNotFoundError("Parent posts required for R2")

## R1 — JSONL to CSV

In [ ]:
R1_OUT = f"{REPLY_DIR}/replies_raw.csv"

if os.path.exists(R1_OUT):
    import pandas as pd
    df = pd.read_csv(R1_OUT)
    print(f"⏭️  Skipping R1 — output exists ({len(df):,} rows)")
else:
    print("Running R1: JSONL → CSV...")
    t0 = time.time()
    !python3 {CODE_DIR}/scripts/replies/r1_jsonl_to_csv.py {REPLY_JSONL}
    elapsed = time.time() - t0
    
    df = pd.read_csv(R1_OUT)
    print(f"✅ R1 done in {elapsed:.0f}s — {len(df):,} rows")

## R2 — Metadata Enrichment

In [ ]:
R2_OUT = f"{REPLY_DIR}/replies_enriched.csv"

if os.path.exists(R2_OUT):
    import pandas as pd
    df = pd.read_csv(R2_OUT)
    print(f"⏭️  Skipping R2 — output exists ({len(df):,} rows)")
else:
    print("Running R2: Metadata enrichment...")
    t0 = time.time()
    
    import pandas as pd
    replies = pd.read_csv(R1_OUT, dtype={"id": str, "parent_id": str})
    parents = pd.read_csv(PARENTS_CSV, dtype={"id": str}, usecols=["id", "created_at", "lang", "date", "hour", "query_raw", "scrape_tab"])
    
    parent_meta = parents.rename(columns={"id": "parent_id", "created_at": "parent_created_at", "lang": "parent_lang", "date": "parent_date", "hour": "parent_hour", "query_raw": "parent_query_raw", "scrape_tab": "parent_scrape_tab"})
    replies = replies.merge(parent_meta, on="parent_id", how="left")
    replies["created_at"] = replies["created_at"].fillna(replies["parent_created_at"])
    replies["lang"] = replies["lang"].fillna(replies["parent_lang"])
    replies["created_at"] = pd.to_datetime(replies["created_at"], errors="coerce")
    replies["date"] = replies["created_at"].dt.date.astype(str)
    replies["hour"] = replies["created_at"].dt.hour
    replies["date"] = replies["date"].fillna(replies["parent_date"])
    replies["hour"] = replies["hour"].fillna(replies["parent_hour"])
    replies = replies.drop(columns=["parent_created_at", "parent_lang", "parent_date", "parent_hour", "parent_query_raw", "parent_scrape_tab"], errors="ignore")
    replies.to_csv(R2_OUT, index=False)
    
    elapsed = time.time() - t0
    print(f"✅ R2 done in {elapsed:.0f}s — {len(replies):,} rows")
    print(f"   Nulls: created_at={replies['created_at'].isna().sum()}, lang={replies['lang'].isna().sum()}")

## R3 — Depth Classification

In [ ]:
R3_OUT = f"{REPLY_DIR}/replies_depth.csv"

if os.path.exists(R3_OUT):
    import pandas as pd
    df = pd.read_csv(R3_OUT)
    print(f"⏭️  Skipping R3 — output exists ({len(df):,} rows)")
else:
    print("Running R3: Depth classification...")
    t0 = time.time()
    
    import pandas as pd
    replies = pd.read_csv(R2_OUT, dtype={"id": str, "parent_id": str})
    parents = pd.read_csv(PARENTS_CSV, dtype={"id": str}, usecols=["id"])
    
    parent_ids = set(parents["id"].astype(str))
    reply_ids = set(replies["id"].astype(str))
    
    def classify_depth(pid):
        pid = str(pid)
        if pid in parent_ids:
            return 1
        elif pid in reply_ids:
            return 2
        return 0
    
    replies["depth"] = replies["parent_id"].apply(classify_depth)
    replies.to_csv(R3_OUT, index=False)
    
    elapsed = time.time() - t0
    print(f"✅ R3 done in {elapsed:.0f}s — {len(replies):,} rows")
    print(replies["depth"].value_counts().to_string())

## R4 — Text Filtering

In [ ]:
R4_OUT = f"{REPLY_DIR}/replies_filtered.csv"

if os.path.exists(R4_OUT):
    import pandas as pd
    df = pd.read_csv(R4_OUT)
    print(f"⏭️  Skipping R4 — output exists ({len(df):,} rows)")
else:
    print("Running R4: Text filtering...")
    t0 = time.time()
    !python3 {CODE_DIR}/scripts/replies/r4_filter_text.py {R3_OUT}
    elapsed = time.time() - t0
    
    df = pd.read_csv(R4_OUT)
    print(f"✅ R4 done in {elapsed:.0f}s — {len(df):,} rows")

## R5 — Language Detection

In [ ]:
R5_OUT = f"{REPLY_DIR}/replies_tagged.csv"

if os.path.exists(R5_OUT):
    import pandas as pd
    df = pd.read_csv(R5_OUT)
    print(f"⏭️  Skipping R5 — output exists ({len(df):,} rows)")
else:
    print("Running R5: Language detection (~30-60s)...")
    t0 = time.time()
    !python3 {CODE_DIR}/scripts/replies/r5_tag_language.py {R4_OUT}
    elapsed = time.time() - t0
    
    df = pd.read_csv(R5_OUT)
    print(f"✅ R5 done in {elapsed:.0f}s — {len(df):,} rows")
    print(df["detected_lang"].value_counts().head().to_string())

## R6 — Text Preprocessing

In [ ]:
R6_OUT = f"{REPLY_DIR}/replies_preprocessed.csv"

if os.path.exists(R6_OUT):
    import pandas as pd
    df = pd.read_csv(R6_OUT)
    print(f"⏭️  Skipping R6 — output exists ({len(df):,} rows)")
else:
    print("Running R6: Text preprocessing (~2-5 min)...")
    t0 = time.time()
    !python3 {CODE_DIR}/scripts/replies/r6_preprocess_text.py {R5_OUT}
    elapsed = time.time() - t0
    
    df = pd.read_csv(R6_OUT)
    print(f"✅ R6 done in {elapsed/60:.1f} min — {len(df):,} rows")

## R7 — Sentiment Analysis (GPU, ~30-60 min)

In [ ]:
R7_OUT = f"{REPLY_DIR}/replies_sentiment.csv"

if os.path.exists(R7_OUT):
    import pandas as pd
    df = pd.read_csv(R7_OUT)
    print(f"⏭️  Skipping R7 — output exists ({len(df):,} rows)")
else:
    print("Running R7: Sentiment analysis (GPU, ~30-60 min for 450k replies)...")
    t0 = time.time()
    !python3 {CODE_DIR}/scripts/replies/r7_sentiment.py {R6_OUT}
    elapsed = time.time() - t0
    
    df = pd.read_csv(R7_OUT)
    print(f"\n✅ R7 done in {elapsed/60:.1f} min — {len(df):,} rows")
    print("\nSentiment distribution:")
    print(df["sentiment_normalized"].value_counts().to_string())

## R8 — Reply Tree & Corpus Combined

In [ ]:
R8_TREE = f"{REPLY_DIR}/reply_tree.csv"
R8_COMBINED = f"{REPLY_DIR}/corpus_combined.csv"

if os.path.exists(R8_TREE) and os.path.exists(R8_COMBINED):
    tree = pd.read_csv(R8_TREE)
    combined = pd.read_csv(R8_COMBINED)
    print(f"⏭️  Skipping R8 — outputs exist")
    print(f"   Reply tree: {len(tree):,} rows")
    print(f"   Corpus combined: {len(combined):,} rows")
else:
    print("Running R8: Building reply tree and combined corpus...")
    t0 = time.time()
    
    import pandas as pd
    
    replies = pd.read_csv(R7_OUT)
    
    parents_sent = None
    if os.path.exists(PARENTS_SENTIMENT):
        parents_sent = pd.read_csv(PARENTS_SENTIMENT, dtype={"id": str},
            usecols=["id", "sentiment_normalized", "topic_id", "engagement_total"])
        print(f"   Parent sentiment: {len(parents_sent):,} rows")
    else:
        print("   ⚠️  Parent sentiment not found — tree will lack parent sentiment")
    
    tree = replies[["id", "parent_id", "depth", "sentiment_normalized",
                    "sentiment_score", "text", "date", "detected_lang",
                    "favorite_count", "retweet_count", "reply_count",
                    "engagement_total"]].copy()
    tree.columns = [
        "reply_id", "parent_id", "depth", "reply_sentiment",
        "reply_sentiment_score", "reply_text", "reply_date", "reply_lang",
        "reply_favorites", "reply_retweets", "reply_replies",
        "reply_engagement"
    ]
    
    if parents_sent is not None:
        parent_sent = parents_sent.rename(columns={
            "id": "parent_id",
            "sentiment_normalized": "parent_sentiment",
            "topic_id": "parent_topic_id",
            "engagement_total": "parent_engagement"
        })
        tree = tree.merge(parent_sent, on="parent_id", how="left")
    
    tree.to_csv(R8_TREE, index=False)
    print(f"✅ Reply tree: {len(tree):,} rows → {R8_TREE}")
    
    shared_cols = [
        "id", "text", "text_clean_light", "text_clean_topic",
        "created_at", "date", "hour", "lang", "detected_lang",
        "favorite_count", "retweet_count", "reply_count",
        "engagement_total", "sentiment_normalized",
        "sentiment_score", "tweet_type"
    ]
    
    replies["tweet_type"] = "reply"
    replies_aligned = replies.reindex(
        columns=shared_cols + ["parent_id", "depth", "parent_engagement_total"]
    )
    
    if os.path.exists(PARENTS_CSV):
        parents = pd.read_csv(PARENTS_CSV, dtype={"id": str})
        parents["tweet_type"] = "parent"
        parents_aligned = parents.reindex(
            columns=shared_cols + ["topic_id", "topic_prob", "query_raw", "scrape_tab"]
        )
        combined = pd.concat([parents_aligned, replies_aligned], ignore_index=True)
    else:
        combined = replies_aligned.copy()
    
    combined.to_csv(R8_COMBINED, index=False)
    
    elapsed = time.time() - t0
    print(f"\n✅ R8 done in {elapsed:.0f}s")
    print(f"   Combined corpus: {len(combined):,} rows")
    print(f"   Parents: {(combined['tweet_type']=='parent').sum():,}")
    print(f"   Replies: {(combined['tweet_type']=='reply').sum():,}")

## R9 — Reply Analysis

In [ ]:
R9_FILES = [
    f"{ANALYSIS_DIR}/reply_controversy_scores.csv",
    f"{ANALYSIS_DIR}/reply_sentiment_shift.csv",
    f"{ANALYSIS_DIR}/reply_depth_sentiment.csv",
    f"{ANALYSIS_DIR}/reply_talk_amplify.csv",
    f"{ANALYSIS_DIR}/reply_most_replied_parents.csv",
]

if all(os.path.exists(f) for f in R9_FILES):
    print(f"⏭️  Skipping R9 — all analysis outputs exist")
else:
    print("Running R9: Reply-specific analysis...")
    t0 = time.time()
    
    import pandas as pd
    
    tree = pd.read_csv(R8_TREE)
    replies = pd.read_csv(R7_OUT)
    
    print(f"   Tree: {len(tree):,} rows | Replies: {len(replies):,} rows")
    
    # ── 1. Controversy scores ──────────────────────────────────
    def calc_controversy(series):
        vc = series.value_counts(normalize=True)
        pos = vc.get("positive", 0)
        neg = vc.get("negative", 0)
        return min(pos, neg) * 2
    
    controversy = tree.groupby("parent_id")["reply_sentiment"].agg(
        controversy_score=calc_controversy,
        reply_count="count"
    ).reset_index()
    controversy.to_csv(R9_FILES[0], index=False)
    print(f"✅ Controversy scores: {len(controversy):,} parent posts")
    
    # ── 2. Sentiment shift parent → reply ─────────────────────
    if "parent_sentiment" in tree.columns:
        shift = tree[["parent_id", "parent_sentiment", "reply_sentiment", "depth"]].dropna()
        shift["sentiment_shift"] = shift.apply(
            lambda r: "same" if r["parent_sentiment"] == r["reply_sentiment"]
            else f"{r['parent_sentiment']}→{r['reply_sentiment']}",
            axis=1
        )
        shift_summary = shift.groupby(["depth", "sentiment_shift"]).size().reset_index(name="count")
        shift_summary.to_csv(R9_FILES[1], index=False)
        print(f"✅ Sentiment shift: {len(shift_summary):,} combinations")
    else:
        print("⚠️  Skipping sentiment shift — no parent_sentiment in tree")
    
    # ── 3. Depth vs sentiment ──────────────────────────────────
    depth_sent = replies.groupby(["depth", "sentiment_normalized"]).size().reset_index(name="count")
    depth_sent.to_csv(R9_FILES[2], index=False)
    print(f"✅ Depth vs sentiment")
    print(depth_sent.to_string(index=False))
    
    # ── 4. Talk vs amplify ratio ───────────────────────────────
    talk_amplify = tree.groupby("parent_id").agg(
        total_replies=("reply_id", "count"),
        parent_engagement=("reply_engagement", "first")
    ).reset_index()
    talk_amplify["talk_amplify_ratio"] = (
        talk_amplify["total_replies"] /
        (talk_amplify["parent_engagement"].fillna(0) + 1)
    )
    talk_amplify.to_csv(R9_FILES[3], index=False)
    print(f"✅ Talk vs amplify ratio: {len(talk_amplify):,} parent posts")
    
    # ── 5. Most replied parents ────────────────────────────────
    most_replied = (
        tree.groupby("parent_id")
        .agg(
            reply_count=("reply_id", "count"),
            avg_reply_sentiment=("reply_sentiment_score", "mean"),
            controversy=("reply_sentiment", lambda s: min(
                (s == "positive").mean(), (s == "negative").mean()
            ) * 2)
        )
        .sort_values("reply_count", ascending=False)
        .head(50)
        .reset_index()
    )
    most_replied.to_csv(R9_FILES[4], index=False)
    print(f"✅ Most replied parents: top 50")
    
    elapsed = time.time() - t0
    print(f"\n✅ R9 done in {elapsed:.0f}s")

## R10 — Upload to DO Spaces

In [ ]:
from google.colab import userdata

s3cfg = f'[default]\naccess_key = {userdata.get("DO_ACCESS_KEY")}\nsecret_key = {userdata.get("DO_SECRET_KEY")}\nhost_base = sgp1.digitaloceanspaces.com\nhost_bucket = %(bucket)s.sgp1.digitaloceanspaces.com\n'
open('/root/.s3cfg', 'w').write(s3cfg)

BUCKET = "s3://mbg-scraper-network-20260419071440/output"

upload_files = [
    (R7_OUT, "replies_with_sentiment.csv"),
    (R8_TREE, "reply_tree.csv"),
    (R8_COMBINED, "corpus_combined.csv"),
]

for src, name in upload_files:
    if os.path.exists(src):
        size_mb = os.path.getsize(src) / 1e6
        print(f"Uploading {name} ({size_mb:.0f}MB)...")
        !s3cmd put {src} {BUCKET}/{name}
        print(f"  ✅ {name}")
    else:
        print(f"  ⚠️  {name} not found — skipping")

print(f"\nUploading analysis files...")
for f in R9_FILES:
    if os.path.exists(f):
        name = os.path.basename(f)
        !s3cmd put {f} {BUCKET}/analysis/{name}
        print(f"  ✅ {name}")

print(f"\n✅ All outputs uploaded to {BUCKET}")

## Pipeline Complete — Summary

In [ ]:
import pandas as pd

total_time = (time.time() - notebook_start_time) / 60

print("=" * 50)
print("MBG REPLY PIPELINE COMPLETE")
print("=" * 50)
print(f"Total runtime: {total_time:.1f} minutes")

df_final = pd.read_csv(R7_OUT)
print(f"\n📊 Reply Sentiment ({len(df_final):,} rows)")
print(df_final["sentiment_normalized"].value_counts().to_string())

print(f"\n📊 Depth Breakdown")
print(df_final["depth"].value_counts().to_string())

if os.path.exists(R8_TREE):
    tree = pd.read_csv(R8_TREE)
    print(f"\n📊 Reply Tree: {len(tree):,} parent-reply links")
    if "parent_sentiment" in tree.columns:
        same = (tree["parent_sentiment"] == tree["reply_sentiment"]).sum()
        print(f"   Same sentiment: {same:,} ({same/len(tree)*100:.1f}%)")

if os.path.exists(R9_FILES[0]):
    cont = pd.read_csv(R9_FILES[0])
    print(f"\n📊 Controversy Scores: {len(cont):,} parent posts")
    print(f"   Avg controversy: {cont['controversy_score'].mean():.3f}")
    print(f"   Max controversy: {cont['controversy_score'].max():.3f}")

print(f"\n✅ Ready for dashboard integration")